![](https://1.bp.blogspot.com/-E_jwjw4zI9A/X9it-nb3cyI/AAAAAAAAHb8/NyNa2Mevt8E8jWLOBBV5ToweaRyYVE9VwCLcBGAsYHQ/s1200/restaurant.jpg)

### Consider this: Chicago, a city with nearly three million people and more than 15,000 food establishments, has fewer than three dozen inspectors who are in charge of annually checking the city’s entire lot. 

### When inspectors check this entire lot, 15% of these establishments, on average, earn a critical violation. 

### Having a critical violation, which generally relates to food temperature control, can drastically increase the odds that a restaurant may start or spread a foodborne illness.   Because of the obvious negative effects this can have on a population, efficiently and effectively targeting food establishments with critical violations is a top public health priority. 

### Chicago’s challenging task to quickly locate and address these violations is a prime candidate for optimization with advanced analytics.  It’s also an opportunity that Chicago’s analytics team has been sure to seize as the City pioneers in its use of data. 

### The City’s recently completed pilot program to optimize the city’s food inspections process – conducted by the Chicago Department of Innovation and Technology (DoIT), along with the Department of Public Health (CDPH) and research partnerships with Civic Consulting Alliance and Allstate Insurance – has been a milestone that has yielded striking results. When using an analytics-based procedure, Chicago was able to discover critical violations, on average, seven days earlier than if they had used the traditional inspection procedure. 

### The results have implications not only for Chicago, but for cities anywhere that wish to optimize inspections processes using advanced analytics.  Moreover, Chicago’s collaborative and open method for launching such an initiative provides lessons for other places that wish to start analytics programs of their own.    

### In processing and analyzing the data, Chicago found several key predicting variables that, when observed, indicated there could be a considerable likelihood that a restaurant may earn a critical violation.  These predicting variables include the following:

* Prior history of critical violations
* Possession of a tobacco and/or incidental alcohol consumption license 
* Length of time establishment has been operating
* Length of time since last inspection
* Location of establishment
* Nearby garbage and sanitation complaints
* Nearby burglaries
* Three day average high temperature

### These predictors were then factored together into a model, which was tested against food inspection procedures via a double-blind post-diction analysis.  In other words, after collecting a set of data, Chicago performed a simulation that used this past data to predict what its future outcome would have been under data-optimized conditions. 




In [ ]:
# Imports
import pandas as pd
import numpy as np
import seaborn as sns
import datetime
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go
%matplotlib inline
from sklearn.model_selection import train_test_split


import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
pd.set_option('display.max_columns', 500)
pd.set_option('display.max_rows', 500)
import os
import gc
gc.enable()
import time
import warnings
warnings.filterwarnings("ignore")
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score
from scipy.sparse import hstack
from scipy import stats
%matplotlib inline
from datetime import timedelta
import datetime as dt
import matplotlib.pyplot as plt
plt.rcParams['figure.figsize'] = [16, 10]
from sklearn.model_selection import train_test_split
from sklearn.decomposition import PCA
from sklearn.cluster import MiniBatchKMeans
import warnings
warnings.filterwarnings('ignore')
import urllib        #for url stuff

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.feature_extraction import text
from IPython.display import display
from tqdm import tqdm
from collections import Counter
import ast


from sklearn.feature_extraction.text import CountVectorizer
from textblob import TextBlob
import scipy.stats as stats

import seaborn as sns
from sklearn.manifold import TSNE
from sklearn.decomposition import PCA, TruncatedSVD
import matplotlib.patches as mpatches
import time

import seaborn as sns #for making plots
import matplotlib.pyplot as plt # for plotting
import os  # for os commands

import gensim
from gensim import corpora, models, similarities
import logging
import tempfile
from nltk.corpus import stopwords
from string import punctuation
from collections import OrderedDict

from sklearn.decomposition import TruncatedSVD
from sklearn.decomposition import LatentDirichletAllocation
from sklearn.manifold import TSNE


from bokeh.plotting import figure, output_file, show
from bokeh.models import Label
from bokeh.io import output_notebook
output_notebook()


!pip install chart_studio
import plotly
import chart_studio.plotly as py
from plotly.offline import init_notebook_mode, iplot
init_notebook_mode(connected=True)
import plotly.graph_objs as go

import warnings
warnings.filterwarnings('ignore')

# Data import
df = pd.read_csv("../input/food-inspections-in-chicago/Food_Inspections.csv")

# Install folium for map visualization

In [ ]:
%%capture
!pip install -U folium

# Import necessary packages 
from folium import folium, plugins
from IPython.display import HTML

%matplotlib inline

In [ ]:
df.head()

In [ ]:
df.rename(columns={"License #": "license"}, inplace=True)
# Extract day, month and year from Inspection Date column
#pandas datetimeindex docs: https://pandas.pydata.org/pandas-docs/stable/generated/pandas.DatetimeIndex.html
#efficient way to extract year from string format date
df['year'] = pd.DatetimeIndex(df['Inspection Date']).year
df['month'] = pd.DatetimeIndex(df['Inspection Date']).month
df['day'] = pd.DatetimeIndex(df['Inspection Date']).day

In [ ]:
df.head()

# Focus is only on “Canvass” inspections

In [ ]:
df = df[df['Inspection Type'].notna()]
df = df[df['Inspection Type']=='Canvass']

df.shape

# Different data types in the dataset

In [ ]:
df.dtypes

# Drop rows with missing data

In [ ]:
df.dropna(subset=["Inspection Date", "license", "Latitude", "Longitude"], inplace=True)

# Only consider successful inspections

In [ ]:
df = df[~df.Results.isin(["Out of Business", "Business Not Located", "No Entry", "Not Ready"])]

# What is DBA name?
#### In the U.S., a DBA lets the public know who the real owner of a business is. The DBA is also called a fictitious business name or assumed business name. It got its origins as a form of consumer protection, so dishonest business owners couldn't try to avoid legal trouble by operating under a different name.

# Number of Different DBA names

In [ ]:
len(set(df['DBA Name'].tolist()))

# Top 10 DBA Names with their distribution of inspections

In [ ]:
df['DBA Name'].value_counts()[:10]

In [ ]:
data_risk1=df
fig,ax=plt.subplots(1,2,figsize=(15,8))
sns.barplot(x=data_risk1['DBA Name'].value_counts()[:10],y=data_risk1['DBA Name'].value_counts()[:10].index,ax=ax[0])
ax[0].set_title("Top 10 Facility Type by the counts of risk",size=20)
ax[0].set_xlabel('counts',size=18)


count=data_risk1.groupby(['DBA Name'])['Inspection ID'].agg('count').sort_values(ascending=False)
groups=list(data_risk1.groupby(['DBA Name'])['Inspection ID'].agg('count').sort_values(ascending=False).index[:10])
counts=list(count[:10])
counts.append(count.agg(sum)-count[:10].agg('sum'))
groups.append('Other')
type_dict=pd.DataFrame({"group":groups,"counts":counts})
clr1=('brown','darksalmon','orange','hotpink','cadetblue','purple','red','gold','forestgreen','blue','plum')
type_dict.plot(kind='pie', y='counts', labels=groups,colors=clr1,autopct='%1.1f%%', pctdistance=0.9, radius=1.2,ax=ax[1])
ax[1].set_ylabel('')
ax[1].legend(loc=0, ncol=1, fontsize=14,bbox_to_anchor=(1.15,1.2))

In [ ]:
fig,ax=plt.subplots(2,2,figsize=(20,16))
y=df['DBA Name'].value_counts()[:10].index
x=df['DBA Name'].value_counts()[:10]
sns.barplot(x=x,y=y,ax=ax[0,0])
ax[0,0].set_title("Top 10 DBA Name by the counts of inspection ",size=20)
ax[0,0].set_xlabel('counts',size=18)
ax[0,0].set_ylabel('')

sns.scatterplot(x='Longitude',y='Latitude',hue='Risk',hue_order=['Risk 1 (High)','Risk 2 (Medium)','Risk 3 (Low)'] ,data=df[df['DBA Name']=='SUBWAY'], ax=ax[0,1])
ax[0,1].set_title("The distribution of inspections for SUBWAY",size=20)
ax[0,1].set_xlabel('Longitude')
ax[0,1].set_ylabel('LATITUDE')

sns.scatterplot(x='Longitude',y='Latitude',hue='Risk' ,hue_order=['Risk 1 (High)','Risk 2 (Medium)','Risk 3 (Low)'],data=df[df['DBA Name']=='DUNKIN DONUTS'], ax=ax[1,0])
ax[1,0].set_title("The distribution of inspections for DUNKIN DONUTS",size=20)
ax[1,0].set_xlabel('Longitude')
ax[1,0].set_ylabel('LATITUDE')

sns.scatterplot(x='Longitude',y='Latitude',hue='Risk',hue_order=['Risk 1 (High)','Risk 2 (Medium)','Risk 3 (Low)'] ,data=df[df['DBA Name']=='7-ELEVEN'], ax=ax[1,1])
ax[1,1].set_title("The distribution of inspections for 7-ELEVEN",size=20)
ax[1,1].set_xlabel('Longitude')
ax[1,1].set_ylabel('LATITUDE')

# Number of Different types of licences

In [ ]:
len(set(df['license'].tolist()))

# Number of Different types of Facilities

In [ ]:
len(set(df['Facility Type'].tolist()))

# Top 10 Facilities inspected

In [ ]:
from plotly.offline import download_plotlyjs, init_notebook_mode, plot, iplot

x = df['Facility Type'].value_counts().index.values.astype('str')[:10]
y = df['Facility Type'].value_counts().values[:10]
pct = [("%.2f"%(v*100))+"%"for v in (y/len(df))][:10]


trace1 = go.Bar(x=x, y=y, text=pct)
layout = dict(title= 'Number of Facility Type',
              yaxis = dict(title='Count'),
              xaxis = dict(title='Facility Type'))
fig=dict(data=[trace1], layout=layout)
iplot(fig)

# Visualization for Facility Type based on Risk type

In [ ]:
fig,ax=plt.subplots(2,2,figsize=(20,16))
y=df['Facility Type'].value_counts()[:10].index
x=df['Facility Type'].value_counts()[:10]
sns.barplot(x=x,y=y,ax=ax[0,0])
ax[0,0].set_title("Top 10 Facility Type by the counts of inspection ",size=20)
ax[0,0].set_xlabel('counts',size=18)
ax[0,0].set_ylabel('')

sns.scatterplot(x='Longitude',y='Latitude',hue='Risk',hue_order=['Risk 1 (High)','Risk 2 (Medium)','Risk 3 (Low)'] ,data=df[df['Facility Type']=='Restaurant'], ax=ax[0,1])
ax[0,1].set_title("The distribution of inspections for restaurant",size=20)
ax[0,1].set_xlabel('Longitude')
ax[0,1].set_ylabel('LATITUDE')

sns.scatterplot(x='Longitude',y='Latitude',hue='Risk' ,hue_order=['Risk 1 (High)','Risk 2 (Medium)','Risk 3 (Low)'],data=df[df['Facility Type']=='Grocery Store'], ax=ax[1,0])
ax[1,0].set_title("The distribution of inspections for Grocery Store",size=20)
ax[1,0].set_xlabel('Longitude')
ax[1,0].set_ylabel('LATITUDE')

sns.scatterplot(x='Longitude',y='Latitude',hue='Risk',hue_order=['Risk 1 (High)','Risk 2 (Medium)','Risk 3 (Low)'] ,data=df[df['Facility Type']=='School'], ax=ax[1,1])
ax[1,1].set_title("The distribution of inspections for School",size=20)
ax[1,1].set_xlabel('Longitude')
ax[1,1].set_ylabel('LATITUDE')

# Risk Types Count

In [ ]:
x = df['Risk'].value_counts().index.values.astype('str')
y = df['Risk'].value_counts().values
pct = [("%.2f"%(v*100))+"%"for v in (y/len(df))]


trace1 = go.Bar(x=x, y=y, text=pct)
layout = dict(title= 'Type of Risk Count',
              yaxis = dict(title='Count'),
              xaxis = dict(title='Risk'))
fig=dict(data=[trace1], layout=layout)
iplot(fig)

### Now let's look on type of risk. The are three types of risk: High, Medium and Low (numerically 1, 2 and 3). Most (over 70%) audited spots had high risk. Second one is medium with around 10% participation and the last one is low risk.

# Crosstab for the results each Month-Day-Year

In [ ]:
pd.crosstab([df.month,df.day],[df.Results,df.year],margins=True).style.background_gradient(cmap='summer_r')

In [ ]:
df.groupby('year').Risk.value_counts().unstack().plot.barh()

In [ ]:
df.groupby('month').Risk.value_counts().unstack().plot.barh()

In [ ]:
df.groupby('year').Results.value_counts().unstack().plot.barh()

# Visualization for Risk 1 (High)

In [ ]:
data_risk1=df[df.Risk=='Risk 1 (High)']
fig,ax=plt.subplots(1,2,figsize=(15,8))
sns.barplot(x=data_risk1['Facility Type'].value_counts()[:10],y=data_risk1['Facility Type'].value_counts()[:10].index,ax=ax[0])
ax[0].set_title("Top 10 Facility Type by the counts of risk 1 ",size=20)
ax[0].set_xlabel('counts',size=18)


count=data_risk1.groupby(['Facility Type'])['Inspection ID'].agg('count').sort_values(ascending=False)
groups=list(data_risk1.groupby(['Facility Type'])['Inspection ID'].agg('count').sort_values(ascending=False).index[:10])
counts=list(count[:10])
counts.append(count.agg(sum)-count[:10].agg('sum'))
groups.append('Other')
type_dict=pd.DataFrame({"group":groups,"counts":counts})
clr1=('brown','darksalmon','orange','hotpink','cadetblue','purple','red','gold','forestgreen','blue','plum')
type_dict.plot(kind='pie', y='counts', labels=groups,colors=clr1,autopct='%1.1f%%', pctdistance=0.9, radius=1.2,ax=ax[1])
ax[1].set_ylabel('')
ax[1].legend(loc=0, ncol=1, fontsize=14,bbox_to_anchor=(1.15,1.2))

In [ ]:
import folium 

data_risk1=df[df.Risk=='Risk 1 (High)']

data_risk1_2000=data_risk1[:2000]
Long=data_risk1_2000.Longitude.mean()
Lat=data_risk1_2000.Latitude.mean()
risk1_map=folium.Map([Lat,Long],zoom_start=12)

risk1_distribution_map=plugins.MarkerCluster().add_to(risk1_map)
for lat,lon,label in zip(data_risk1_2000.Latitude,data_risk1_2000.Longitude,data_risk1_2000['AKA Name']):
    folium.Marker(location=[lat,lon],icon=None,popup=label).add_to(risk1_distribution_map)
risk1_map.add_child(risk1_distribution_map)

risk1_map

# Visualization for Risk 2 (Medium)

In [ ]:
data_risk2=df[df.Risk=='Risk 2 (Medium)']

fig,ax=plt.subplots(1,2,figsize=(15,8))
sns.barplot(x=data_risk2['Facility Type'].value_counts()[:10],y=data_risk2['Facility Type'].value_counts()[:10].index,ax=ax[0])
ax[0].set_title("Top 10 Facility Type by the counts of risk 2 ",size=20)
ax[0].set_xlabel('counts',size=18)


count=data_risk2.groupby(['Facility Type'])['Inspection ID'].agg('count').sort_values(ascending=False)
groups=list(data_risk2.groupby(['Facility Type'])['Inspection ID'].agg('count').sort_values(ascending=False).index[:10])
counts=list(count[:10])
counts.append(count.agg(sum)-count[:10].agg('sum'))
groups.append('Other')
type_dict=pd.DataFrame({"group":groups,"counts":counts})
clr1=('brown','darksalmon','orange','hotpink','cadetblue','purple','red','gold','forestgreen','blue','plum')
type_dict.plot(kind='pie', y='counts', labels=groups,colors=clr1,autopct='%1.1f%%', pctdistance=0.9, radius=1.2,ax=ax[1])
ax[1].set_ylabel('')
ax[1].legend(loc=0, ncol=1, fontsize=14,bbox_to_anchor=(1.15,1.2))

In [ ]:
data_risk2_2000=data_risk2[:2000]
Long=data_risk2_2000.Longitude.mean()
Lat=data_risk2_2000.Latitude.mean()
risk2_map=folium.Map([Lat,Long],zoom_start=12)

risk2_distribution_map=plugins.MarkerCluster().add_to(risk2_map)
for lat,lon,label in zip(data_risk2_2000.Latitude,data_risk2_2000.Longitude,data_risk2_2000['AKA Name']):
    folium.Marker(location=[lat,lon],icon=None,popup=label).add_to(risk2_distribution_map)
risk2_map.add_child(risk2_distribution_map)

risk2_map

# Visualization for Risk 3 (Low)

In [ ]:
data_risk3=df[df.Risk=='Risk 3 (Low)']

fig,ax=plt.subplots(1,2,figsize=(15,8))
sns.barplot(x=data_risk3['Facility Type'].value_counts()[:10],y=data_risk3['Facility Type'].value_counts()[:10].index,ax=ax[0])
ax[0].set_title("Top 10 Facility Type by the counts of risk 3 ",size=20)
ax[0].set_xlabel('counts',size=18)


count=data_risk3.groupby(['Facility Type'])['Inspection ID'].agg('count').sort_values(ascending=False)
groups=list(data_risk3.groupby(['Facility Type'])['Inspection ID'].agg('count').sort_values(ascending=False).index[:10])
counts=list(count[:10])
counts.append(count.agg(sum)-count[:10].agg('sum'))
groups.append('Other')
type_dict=pd.DataFrame({"group":groups,"counts":counts})
clr1=('brown','darksalmon','orange','hotpink','cadetblue','purple','red','gold','forestgreen','blue','plum')
type_dict.plot(kind='pie', y='counts', labels=groups,colors=clr1,autopct='%1.1f%%', pctdistance=0.9, radius=1.2,ax=ax[1])
ax[1].set_ylabel('')
ax[1].legend(loc=0, ncol=1, fontsize=14,bbox_to_anchor=(1.15,1.2))

In [ ]:
data_risk3_2000=data_risk3[:2000]
Long=data_risk3_2000.Longitude.mean()
Lat=data_risk3_2000.Latitude.mean()
risk3_map=folium.Map([Lat,Long],zoom_start=12)

risk3_distribution_map=plugins.MarkerCluster().add_to(risk3_map)
for lat,lon,label in zip(data_risk3_2000.Latitude,data_risk3_2000.Longitude,data_risk3_2000['AKA Name']):
    folium.Marker(location=[lat,lon],icon=None,popup=label).add_to(risk3_distribution_map)
risk3_map.add_child(risk3_distribution_map)

risk3_map

# Types of Risk Analysis Plot

## I'll plot all facilities on the map of Chicago based on the colors that I define.

### The first step is to identify all facilities and take the recent inspections for each facility. I'll also remove all rows where 'Risk', 'Facility Type', 'DBA Name', 'Latitude', 'Longitude' will have null value. Some businesses are no longer operating or are no longer located and thus can be removed too. I'll create a new column Name which extracts the name from AKA Name and DBA Name with preference given to AKA Name.

In [ ]:
latest_data = df.sort_values('Inspection Date', ascending = False).groupby('license').head(1)
latest_data.dropna(subset=['Risk', 'Facility Type', 'DBA Name', 'Latitude', 'Longitude'], axis = 0, how = 'all', inplace = True)
latest_data = latest_data[(latest_data['Results'] != 'Out of Business') & (latest_data['Results'] != 'Business Not Located')]
latest_data['Name'] = latest_data.apply(lambda row: row['AKA Name'] if not pd.isnull(row['AKA Name']) else row['DBA Name'], axis = 1)
latest_data['Name'] = latest_data['Name'] + '<br>' + latest_data['Address']

## Create a Risk Color column which will help in plotting colors for each facility based on Risk.

1. All -> Black
2. High Risk -> Red
3. Medium Risk -> Yellow
4. Low Risk -> Green

## For inspections, I'll crate the Inspection Color column.

1. Pass or Pass w/ Conditions -> Green
2. Fail or No Entry or Not Ready -> Red

In [ ]:
risk_color_map = { "All": "rgb(0, 0, 0)", "Risk 1 (High)": "rgb(255, 0, 0)", "Risk 2 (Medium)": "rgb(204, 204, 0)", "Risk 3 (Low)": "rgb(0, 100, 0)" }
latest_data['Risk Color'] = latest_data['Risk'].map(risk_color_map)

inspection_color_map = { 
    "Pass": "rgb(0, 255, 0)", 
    "Pass w/ Conditions": "rgb(0, 255, 0)",
    "Fail": "rgb(255, 0, 0)", 
    "No Entry": "rgb(255, 0, 0)", 
    "Not Ready": "rgb(255, 0, 0)" }
latest_data['Inspection Color'] = latest_data['Results'].map(inspection_color_map)
    
latest_data.reset_index(inplace=True)
print("Total businesses: {}".format(latest_data.shape[0]))

# Risk Analysis

### Plot all facilities on the map of Chicago based on the colors we defined above.

In [ ]:
# Create and account on mapbox.com and get access token
mapbox_access_token = "pk.eyJ1IjoiaGFtZGl0YXJlazAxIiwiYSI6ImNraXl0eG1zODI0dGUydm1tdWoybHFsNmUifQ.JJry5XjNLcMXZTPmeGIKgw"

In [ ]:
data = [
    go.Scattermapbox(
        lat = latest_data['Latitude'],
        lon = latest_data['Longitude'],
        text = latest_data['Name'],
        hoverinfo = 'text',
        mode = 'markers',
        marker = go.scattermapbox.Marker(
            color = latest_data['Risk Color'],
            opacity = 0.7,
            size = 4
        )
    )
]

layout = go.Layout(
    mapbox = dict(
        accesstoken = mapbox_access_token,
        zoom = 10,
        center = dict(
            lat = 41.8781,
            lon = -87.6298
        ),
    ),
    height = 800,
    width = 800,
    title = "Facilities in Chicago")

fig = go.Figure(data, layout)
iplot(fig, filename = 'facilities')


# Go deeper with Results Analysis

## Success and Failure

### Next, we take a look to facilities that passed the inspection and the ones that did not.

### We can consider Pass and Pass w/ Conditions to be positive outcome and the remaining as negative. Taking a look to the previous visualiazation about Results distribuation, we can see that even though there are many facilities with high risk, most pass the inspection none the less. Let's plot these on a map.

In [ ]:
data = [
    go.Scattermapbox(
        lat = latest_data['Latitude'],
        lon = latest_data['Longitude'],
        text = latest_data['Name'],
        hoverinfo = 'text',
        mode = 'markers',
        marker = go.scattermapbox.Marker(
            color = latest_data['Inspection Color'],
            opacity = 0.7,
            size = 4
        )
    )
]

layout = go.Layout(
    mapbox = dict(
        accesstoken = mapbox_access_token,
        zoom = 10,
        center = dict(
            lat = 41.8781,
            lon = -87.6298
        ),
    ),
    height = 800,
    width = 800,
    title = "Facilities in Chicago")

fig = go.Figure(data, layout)
iplot(fig, filename = 'facilities')

# Count of Inspectation Results

In [ ]:
x = df['Results'].value_counts().index.values.astype('str')
y = df['Results'].value_counts().values
pct = [("%.2f"%(v*100))+"%"for v in (y/len(df))]


trace1 = go.Bar(x=x, y=y, text=pct)
layout = dict(title= 'Inspectation Results Count',
              yaxis = dict(title='Count'),
              xaxis = dict(title='Results'))
fig=dict(data=[trace1], layout=layout)
iplot(fig)

### Most food facilities got pass (over 50%). The second one is "Fail" - over 15 thousand don't get positive opinion after control. Popular is also pass with conditions (over 15% spots got that result). Rarerly are situation when doors are not open or place is not ready for control or even not exist.

In [ ]:
df.groupby(['Risk', 'Results']).size().reset_index(name="Frequency")

In [ ]:
sns.heatmap(pd.crosstab([df.Risk], [df.Results]),
            cmap="YlGnBu", annot=True, fmt=".1f", linewidths=1.0, square=1, cbar=False)

### We have information about the risk and types of control - time to compare this information. Most popular is combination pass and high risk (almost half of controls). The second one is fail and high risk. Last combiantion is pass with medium risk. It looks like there is no correlation between these two variables.

# Visualization for Results of inspections

In [ ]:
fig,ax=plt.subplots(2,2,figsize=(20,16))
x=df.Results.value_counts().index
y=df.Results.value_counts()
sns.barplot(x=x,y=y,ax=ax[0,0])
ax[0,0].set_title("The counts of Results of inspection ",size=20)
ax[0,0].set_ylabel('counts',size=18)
ax[0,0].set_xlabel('')

df.groupby(['Results','year'])['Inspection ID'].agg('count').unstack('Results').plot(kind='bar',ax=ax[0,1])
ax[0,1].tick_params(axis='x',labelrotation=360)
ax[0,1].legend(loc=0, ncol=1, fontsize=14,bbox_to_anchor=(1.15,0.75))
ax[0,1].set_title("The counts of results of inspection by year ",size=20)
ax[0,1].set_ylabel('counts',size=18)

sns.scatterplot(x='Longitude',y='Latitude',hue='Risk' ,hue_order=['Risk 1 (High)','Risk 2 (Medium)','Risk 3 (Low)'],data=df[df.Results=='Pass'], ax=ax[1,0])
ax[1,0].set_title("The distribution of result is pass",size=20)
ax[1,0].set_xlabel('Longitude')
ax[1,0].set_ylabel('LATITUDE')

sns.scatterplot(x='Longitude',y='Latitude',hue='Risk',hue_order=['Risk 1 (High)','Risk 2 (Medium)','Risk 3 (Low)'] ,data=df[df.Results=='Fail'], ax=ax[1,1])
ax[1,1].set_title("The distribution of result is fail",size=20)
ax[1,1].set_xlabel('Longitude')
ax[1,1].set_ylabel('LATITUDE')

# Restaurant or no Restaurant

#### Add new feature called Restaurant that describe if the Facility Type is restaurant or not

In [ ]:
df['Restaurant'] = (df['Facility Type'].values == 'Restaurant').astype('int')

In [ ]:
sns.heatmap(pd.crosstab([df.Restaurant], [df.Risk]),
            cmap="YlGnBu", annot=True, fmt=".1f", linewidths=1.0, square=1, cbar=False)

#### Over half examples are restaurant with high risk (more than 50%). High risk is the most popular type of risk in both - restaurants and other. Interesting is distirbution of "low risk". Low risk is almost always in "no restaurant" examples.

# The heatmap between Restaurant and Resulats

In [ ]:
sns.heatmap(pd.crosstab([df.Restaurant], [df.Results]),
            cmap="YlGnBu", annot=True, fmt=".1f", linewidths=1.0, square=1, cbar=False)

# Drop duplicates

In [ ]:
df.drop_duplicates("Inspection ID", inplace=True)

In [ ]:
df.shape

# INSPECTIONS MAP

In [ ]:
# Find minimum and maximum values for latitude and longitude
la = df['Latitude'].tolist()
lo =df['Longitude'].tolist()
print('The minimum value for the latitude is: '+str(min(la)))
print('The maximum value for the latitude is: '+str(max(la)))
print('The minimum value for the longitude is: '+str(min(lo)))
print('The maximum value for the longitude is: '+str(max(lo)))

In [ ]:
m = folium.Map([41.8600, -87.6298], zoom_start=10)

# Convert to (n, 2) nd-array format for heatmap
inspections_arr = df.sample(20000)[["Latitude", "Longitude"]].values

# Plot heatmap
m.add_child(plugins.HeatMap(inspections_arr.tolist(), radius=10))

# NLP for VIOLATIONS Description

#### The data contain violations column which contain the number of violations with comment for each one.

In [ ]:
# Drop rows with "nan" value in violation column
df['Violations'] = df['Violations'].astype(str)
df = df[df.Violations != "nan"]
df.shape

In [ ]:
df.iloc[0].Violations

# Majority violation

### Let's also check the majority violation that is present among the dataset.

In [ ]:
import re
violators = latest_data.dropna(subset=['Violations'], axis = 0, how = 'all')
violations = violators.apply(lambda row: re.findall('\|\s([0-9]+)[.]', str(row['Violations'])), axis = 1)
first_violations = violators.apply(lambda row: row['Violations'].split('.')[0], axis = 1)

for violation, first_violation in zip(violations, first_violations):
    violation.append(first_violation)

flat_list = [item for sublist in violations for item in sublist]
unique, counts = np.unique(flat_list, return_counts=True)

### I'll select the violations that are more than 100 in count.

In [ ]:
violation = []
violation_count = []
for value, count in zip(unique, counts):
    if count > 100:
        violation.append(unique)
        violation_count.append(count)

In [ ]:
data = [
    go.Bar(
        x = violation,
        y = violation_count,
        marker = dict(
            color = 'rgb(55, 83, 109)'
        )
    )
]

layout = go.Layout(
    title = 'Majority Violations',
)

fig = go.Figure(data = data, layout = layout)
iplot(fig, filename = 'violations')

### Violation 3 is the majority violation which refers to MANAGEMENT, FOOD EMPLOYEE AND CONDITIONAL EMPLOYEE; KNOWLEDGE, RESPONSIBILITIES AND REPORTING

# Split violations into binary values for each violation


In [ ]:
def split_violations(violations):
    values_row = pd.Series([])
    if type(violations) == str:
        violations = violations.split(' | ')
        for violation in violations:
            index = "v_" + violation.split('.')[0]
            values_row[index] = 1
    return values_row

# Calculate violation values (5 mins), set missing violations to 0
values_data = df.Violations.apply(split_violations).fillna(0)

# Generate column names
critical_columns = [("v_" + str(num)) for num in range(1, 15)]
serious_columns = [("v_" + str(num)) for num in range(15, 30)]
minor_columns = [("v_" + str(num)) for num in range(30, 45)]
minor_columns.append("v_70")

# Create complete list of column names
columns = critical_columns + serious_columns + minor_columns

# Create dataframe using column names, violation data and inspection ID
values = pd.DataFrame(values_data, columns=columns)
values['Inspection ID'] = df['Inspection ID']

In [ ]:
# Display values dataframe
print(values.shape)
values.head()

# Count violations

In [ ]:
counts = pd.DataFrame({
    "critical_count": values[critical_columns].sum(axis=1),
    "serious_count": values[serious_columns].sum(axis=1),
    "minor_count": values[minor_columns].sum(axis=1)
})

counts['Inspection ID'] = df['Inspection ID']

In [ ]:
# Display counts dataframe
print(counts.shape)
counts.head()

# COMMENTS WORDCLOUD

In [ ]:
%%capture
!pip install wordcloud

In [ ]:
from wordcloud import WordCloud
import matplotlib.pyplot as plt

# Extract comments from violations
def get_comments(violations):
    comments = ""
    if type(violations) == str:
        violations = violations.split(' | ')
        for violation in violations:
            violation = violation.split('Comments:')
            if len(violation) == 2:
                comments += violation[1]
    return comments

# Concatenate all comments
comments = df.Violations.apply(get_comments).str.cat(sep=" ")

# Generate wordcloud
comments_wordcloud = WordCloud().generate(comments)

# Plot wordcloud
plt.rcParams['figure.figsize'] = (10, 10)
plt.imshow(comments_wordcloud, interpolation='bilinear')
plt.axis("off")
plt.show()

### The most occurrence words are PREP, CLEAN, repair, ISSUED, Maintain,...

# VIOLATIONS TITLES

##### The violation_titles.csv file is created in the 21_calculate_violation_data.ipynb file within the [CODE folder](https://github.com/Sustainabilist/ChicagoDataAnalysis/blob/master/CODE/21_calculate_violation_data.ipynb).

In [ ]:
titles = pd.DataFrame({
    "v_1": "Approved food sources (1)",
    "v_2": "Hot/cold storage facilities (2)",
    "v_3": "Hot/cold storage temp. (3)",
    "v_4": "Contaminant protection (4)",
    "v_5": "No sick handlers (5)",
    "v_6": "Proper hand washing (6)",
    "v_7": "Proper utensil washing (7)",
    "v_8": "Proper sanitizing solution (8)",
    "v_9": "Hot/cold water supply (9)",
    "v_10": "Waste water disposal (10)",
    "v_11": "Adequate toilet facilities (11)",
    "v_12": "Adequate hand washing facilities (12)",
    "v_13": "Control of rodents, other pests (13)",
    "v_14": "Correct serious violations (14)",
    "v_15": "No re-served food (15)",
    "v_16": "Protection from contamination (16)",
    "v_17": "Proper thawing (17)",
    "v_18": "Pest control, associated areas (18)",
    "v_19": "Proper garbage area (19)",
    "v_20": "Proper garbage storage (20)",
    "v_21": "Oversight of hazardous food (21)",
    "v_22": "Dishwasher maintenance (22)",
    "v_23": "Scrape before washing (23)",
    "v_24": "Proper dishwashers (24)",
    "v_25": "Minimize toxic materials (25)",
    "v_26": "Adequate customer toilets (26)",
    "v_27": "Supplied toilet facilities (27)",
    "v_28": "Visible inspection report (28)",
    "v_29": "Correct minor violations (29)",
    "v_30": "Labelled containers (30)",
    "v_31": "Sterile utensils (31)",
    "v_32": "Clean, maintain equipment (32)",
    "v_33": "Clean, sanitize utensils (33)",
    "v_34": "Clean, maintain floor (34)",
    "v_35": "Maintain walls & ceiling (35)",
    "v_36": "Proper lighting (36)",
    "v_37": "Toilet rooms vented (37)",
    "v_38": "Proper venting, plumbing (38)",
    "v_39": "Linen, clothing storage (39)",
    "v_40": "Proper thermometers (40)",
    "v_41": "Clean facilities, store supplies (41)",
    "v_42": "Ice handling, hairnets, clothes (42)",
    "v_43": "Ice equipment storage (43)",
    "v_44": "Restrict prep area traffic (44)",
    "v_70": "Restrict smoking (70)"
}, index=[0])

In [ ]:
titles

In [ ]:
# Change the name of columns in value dataframe by the title values dataframe's columns

titled_values = values.rename(columns=titles.iloc[0])

# Sum binary values for each violation
sums = titled_values.drop("Inspection ID", axis=1).sum()

# Generate color list
colors = ["red"]*15 + ["orange"]*14 + ["green"]*16

# Sort sums and colors by sum value
sum_data = pd.DataFrame({"sums": sums, "colors": colors}).sort_values("sums")

# Plot bar chart
plt.rcParams['figure.figsize'] = (10, 10)
ax = sum_data.sums.plot(kind="barh", color=sum_data.colors)
ax.set_title("Health Code Violations", fontsize=25)
ax.set_xlabel("Violation Count", fontsize=15)
ax.invert_yaxis()
plt.show()

### As this chart makes clear, the vast majority of violations are minor (30+) and (3) Hot/cold  storage temp, with only a scattering of serious (15-29) and critical (1-14) violations.

In [ ]:
titled_values

# INSPECTION HISTORY

#### The Chicago team found one of the greatest predictors of critical violations and failed inspections to be the establishment's recent inspection history. To validate this we grouped inspections by license, shifted each group to find the previous inspection and set up a table to compare conditional likelihoods:

In [ ]:
# Sort inspections by date
df = df.sort_values(by="Inspection Date")

# Only consider inspections with clear results
df = df.loc[df.Results.isin(["Pass", "Fail"])]

# Group inspections by license and shift 1 to find previous results
df["previous_results"] = df.groupby(by="license").shift().Results

# Calculate cross tabulation of results and previous results
chart = pd.crosstab(df.previous_results, df.Results)

# Make Numpy array of total counts of prior fails and passes with the 
# following(post) results
chart_arr = np.array(chart)

# Create new dataframe from Numpy array to clearly dispay prior and 
# post results
pass_fail_chart = pd.DataFrame({"Prior Fail":chart_arr[:,0],
                                "Prior Pass":chart_arr[:,1]})
pass_fail_chart.index = pass_fail_chart.index.rename("")
pass_fail_chart = pass_fail_chart.rename(index={0:"Post Fail",1:"Post Pass"})

# Display chart
pass_fail_chart

### Percentage of how may prior fails resulted in post fails

In [ ]:
fail_fail_probability = (pass_fail_chart.loc["Post Fail", "Prior Fail"] /
pass_fail_chart.loc["Post Fail", :].sum())

print(str(100*fail_fail_probability) + " of prior fails resulted in a subsequent fail")

### Percentage of how may prior passes resulted in post fails

In [ ]:
pass_fail_probability = (pass_fail_chart.loc["Post Pass", "Prior Fail"] /
pass_fail_chart.loc["Post Pass", :].sum())

print(str(100*pass_fail_probability)+ " of prior passes resulted in a subsequent fail")

### pass/fail probability vs fail/fail probability

In [ ]:
print(str(100*(1 - (pass_fail_probability / fail_fail_probability))) +" more likely that a prior fail will predict a subsequent fail than a prior pass")

#### We found that facilities with a previous failure were almost twice as likely to fail as those with previous passing inspections, supporting the findings of the Chicago team.

# ECONOMIC IMPACT

In [ ]:
set(df['Risk'].tolist())

In [ ]:
# Create temporary dataframe
temp = pd.merge(df, values, on="Inspection ID")

# Convert inspection_date to datetime format
temp["datetime"] = pd.to_datetime(temp["Inspection Date"])

# Define a function to map fines fees to their values 
def set_value(row_number, assigned_value): 
    return assigned_value[row_number] 
  
# Create the dictionary 
fees_dictionary ={'Risk 1 (High)' : 600, 'Risk 2 (Medium)' : 400, 'Risk 3 (Low)' : 200} 
  
# Add a new column named 'Price' 
temp['fines'] = temp['Risk'].apply(set_value, args =(fees_dictionary, )) 


# Count critical violations
temp["criticals"] = temp[critical_columns].sum(axis=1)

# Display snapshot of temp dataset
temp.head()

### We then grouped inspections by license (a code shared by all inspections for a business) and for each group determined the age, yearly fails and other statistics:

In [ ]:
import math

# Sort by date
temp.sort_values("datetime", inplace=True)

# Calculate statistics for license groups
def get_stats(group):
    days = (group.iloc[-1].datetime - group.iloc[0].datetime).days + 1
    years = days / 365.25
    inspections = len(group)
    yearly_inspections = inspections / math.ceil(years)
    fails = len(group[group.Results == "Fail"])
    yearly_fails = fails / math.ceil(years)
    fines = group.fines.sum()
    yearly_fines = fines / math.ceil(years)
    criticals = group.criticals.sum()
    yearly_criticals = criticals / math.ceil(years)
    return pd.Series({
        "years": years,
        "inspections": inspections,
        "yearly_inspections": yearly_inspections,
        "fails": fails,
        "yearly_fails": yearly_fails,
        "fines": fines,
        "yearly_fines": yearly_fines,
        "criticals": criticals,
        "yearly_criticals": yearly_criticals
    })

# Group by license and apply get_stats
businesses = temp.groupby('license').apply(get_stats).reset_index()

In [ ]:
# Display snapshot of business stats dataset
businesses.head()

#### To assess whether businesses fail enough inspections for risk analysis to be worthwhile we plotted a pareto chart describing the number of businesses in each yearly fails bracket:

In [ ]:
# Define function to plot pareto chart
def plot_pareto(series, title, xlabel, ylabel, line_color):
    index = np.arange(len(series))
    fig, ax1 = plt.subplots()
    ax1.bar(index, series)
    ax1.plot(index, series.cumsum(), color=line_color)
    ax1.set_xticks(index)
    ax1.set_xticklabels(series.index)
    ax1.set_title(title, fontsize=25)
    ax1.set_xlabel(xlabel, fontsize=15)
    ax1.set_ylabel(ylabel, fontsize=15)
    ax2 = ax1.twinx()
    ax2.set_yticks([1, 2, 3, 4, 5, 5.25])
    ax2.set_yticklabels([20 for x in range(1,6)])
    plt.show()

# Count restaurants in each rounded yearly fails bracket
yearly_fail_counts = businesses.yearly_fails.round(1).value_counts()

# Plot pareto chart
plt.rcParams['figure.figsize'] = (10, 7)
plot_pareto(
    yearly_fail_counts,
    "Yearly Failed Inspections",
    "Average Yearly Failed Inspections",
    "Businesses", "orange"
)

In [ ]:
# Fraction of businesses failing .5 or more inspections yearly
len(businesses.loc[businesses.yearly_fails >= .5]) / len(businesses)

In [ ]:
# Fraction of businesses failing 1 or more inspections yearly
len(businesses.loc[businesses.yearly_fails >= 1]) / len(businesses)

### We found that 29% of businesses fail over .5 inspections in a year, with 17% failing 1 or more inspections yearly.

## We then plotted a similar pareto chart to explore the number of businesses in each bracket of yearly critical violations:

In [ ]:
# Count restaurants in each rounded yearly fails bracket
yearly_critical_counts = businesses.yearly_criticals.round().value_counts()

# Plot pareto chart
plot_pareto(
    yearly_critical_counts,
    "Yearly Critical Violations",
    "Average Yearly Critical Violations",
    "Businesses", "red"
)

### As this plot shows, roughly 20% of businesses experience one or more critical violations each year.

### To see if violations could be easily predicted as a function of seasonal temperature, we plotted the percentage of all fines paid for each month:

In [ ]:
# Begin by using the temp dataset to extract the dates of inpections
# Example visual of the date information
temp.datetime[1]

In [ ]:
# Calculate total fines paid for each month
temp["month"] = temp.datetime.apply(lambda x: x.month)
month_fines = temp.groupby("month").fines.sum() / temp.fines.sum()

# List months
months = ["January", "February", "March", "April", "May", "June", "July", "August", "September", "October", "November", "December"]

# Plot bar chart
fig, ax = plt.subplots()
index = np.arange(len(month_fines))
ax.barh(index, month_fines)
ax.set_yticks(index)
ax.set_yticklabels(months, fontsize=15)
ax.set_xticks([x/100 for x in range(5)])
ax.set_xticklabels([x for x in range(5)])
ax.set_title("Fines by Month", fontsize=25)
ax.set_xlabel("Percent of All Fines", fontsize=15)

# Display chart
plt.show()

### Violation fines clearly show strong seasonal variation, with more than twice as much paid in June as in December.

### To explore how much these businesses pay in fines we plotted a histogram of fines paid yearly (600 dollars per critical violation, 200 dollars per serious violation:

In [ ]:
# Count restaurants in each yearly fines bracket
fine_counts = businesses.yearly_fines.round(-2).value_counts().sort_index()

index = np.arange(len(fine_counts))

# Plot bar chart
fig, ax = plt.subplots()
ax.bar(index, fine_counts)
ax.set_xticks(index[::5])
ax.set_xticklabels([x for x in fine_counts.index[::5]])
ax.set_title("Yearly Fines", fontsize=25)
ax.set_xlabel("Yearly Amount Paid in $", fontsize=15)
ax.set_ylabel("Businesses", fontsize=15)

# Display chart
plt.show()

In [ ]:
# Fraction of businesses paying $600 or more yearly
len(businesses.loc[businesses.yearly_fines >= 600]) / len(businesses)

In [ ]:
# Fraction of businesses paying $250 or more yearly
len(businesses.loc[businesses.yearly_fines >= 200]) / len(businesses)

## We found that 53% of businesses pay 600 dollars or more per year in fines and 97% pay 200 dollars or more.

### Finally, we calculated the total amount businesses spent each year on food inspection violations. After plotting the total fines by year, the average was calculated to provide an overall view of the bottom line potential in helping businesses predict and remedy potential violaions.

In [ ]:
# Create line graph showing change in total average yearly fines for all businesses

# Calculate total fines paid for each year

temp["year"] = temp.datetime.apply(lambda x: x.year)
year_fines = temp.groupby("year").fines.sum()


# Caclulate average yearly fines plot line
fines_mean = [np.mean(year_fines)]*len(year_fines)

# Create y axis labels
fines_list = list(year_fines)
fines_list.sort()

# Format fines axis
def millions_format(num, m=1000000):
    if num % m == 0:
        num = int(num/m)
    else:
        num = round(float(num/m), 2)
    return "${} million".format(num)

# Plot line graph
fig, ax = plt.subplots()
ax.plot(year_fines, label="Yearly Fines", marker="D")

# Display average on graph
ax.text(2010, 2000000, r'Average businesses spend per year:')
ax.text(2010.7, 1930000, millions_format(fines_mean[0]))

# Plot the mean line
ax.plot(year_fines.index, fines_mean, label="Mean", linestyle="--")

# Set axis labels and title
ax.set_xlabel("Year", fontsize=15)
ax.set_ylabel("Dollars", fontsize=15)
ax.set_yticklabels([millions_format(number) for number in fines_list])
ax.set_title("Total Fines Per Year", fontsize=25)

# Make a legend
legend = ax.legend(loc='center right')

# Display graph
plt.show()

## Though the chart of fines per year shows an upward trend of fines increasing, there are key factors that can explain this rising cost. Fewer inspections and fewer business licenses will result in less food violation fines. However, as populations grow and weather patterns shift, there will be a higher demand for more food services, more risk for violation 3 (hot and cold food storage at proper temperatures), and overall more opportunities for violations to occur.
## It's clear that there is a big dump in the chart for 2020, I think because of COVID-19 most Businesses are closed, so there are not many inspections during this period.

In [ ]:
# Convert inspection_date to datetime format
df["datetime"] = pd.to_datetime(df["Inspection Date"])

df.sort_values("datetime", inplace=True)

In [ ]:
df.head(50)

# Data preprocessing

In [ ]:
train = pd.merge(df, titled_values, on='Inspection ID')
train = pd.merge(train, counts, on='Inspection ID')

# We will use it to show result and location for next inspectations
# 13452 is the length of test set, we will see it in th next cells
result = train[['DBA Name', 'AKA Name', 'Address', 'Zip', 'Latitude', 'Longitude']].tail(13452)

In [ ]:
print(train.shape)
train.tail(3)

### We need to apply Label Encoding to some features: Label encoding algorithm is quite simple and it considers an order for encoding, Hence can be used for encoding ordinal data.

* DBA Name
* Facility Type
* Address
* Zip
* Risk

In [ ]:
# labelEncoder present in scikitlearn library 

from sklearn.preprocessing import LabelEncoder 

le = LabelEncoder() 
train['DBA Name'] = le.fit_transform(train['DBA Name'])
train['Facility Type'] = train['Facility Type'].astype(str)
train['Facility Type'] = le.fit_transform(train['Facility Type'])
train['Address'] = le.fit_transform(train['Address'])
train['Zip'] = le.fit_transform(train['Zip'])
train['Results'] = (train['Results'].values == 'Fail').astype('int')
train['Risk'] = le.fit_transform(train['Risk'])

In [ ]:
train.head(2)

### Drop unuseful features

* Inspection ID
* AKA Name
* City	
* State	
* Inspection Date	
* Inspection Type
* Violations
* Location

In [ ]:
train = train.drop(['Inspection ID', 'AKA Name', 'City', 'State', 'Inspection Date', 
              'Inspection Type', 'Violations', 'Location', 'previous_results', 'datetime'], axis = 1)

# Modelingfrom sklearn import preprocessing

In [ ]:
import xgboost as xgb
print("XGBoost version:", xgb.__version__)

# Is the data balanced or not?

In [ ]:
x = train['Risk'].value_counts().index
y = train['Risk'].value_counts().values

trace2 = go.Bar(
     x=x ,
     y=y,
     marker=dict(
         color=y,
         colorscale = 'Viridis',
         reversescale = True
     ),
     name="Imbalance",    
 )
layout = dict(
     title="Data imbalance - Risk Type",
     #width = 900, height = 500,
     xaxis=go.layout.XAxis(
     automargin=True),
     yaxis=dict(
         showgrid=False,
         showline=False,
         showticklabels=True,
 #         domain=[0, 0.85],
     ), 
)
fig1 = go.Figure(data=[trace2], layout=layout)
iplot(fig1)

In [ ]:
from sklearn.model_selection import train_test_split
# create dataset
X = train.drop(['Risk'], axis=1)
y = train['Risk']
# split into train test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)
print(X_train.shape, X_test.shape, y_train.shape, y_test.shape)

In [ ]:
clf1 = xgb.XGBClassifier(
    n_estimators=1000,
    max_depth=7,
    learning_rate=0.4,
    #subsample=0.9,
    colsample_bytree=0.6,
    missing=-999,
    random_state=2020,
    tree_method='gpu_hist'  # THE MAGICAL PARAMETER
)


# Fit model
%time clf1.fit(X_train, y_train)
print('//////////'*10)
print('\\\\\\\\\\\\\\\\\\\\'*10)
# make predictions
from sklearn.metrics import accuracy_score

yhat1 = clf1.predict(X_test)

# evaluate predictions
acc = accuracy_score(y_test, yhat1)
print('Accuracy: %.3f' % acc)

print('//////////'*10)
print('\\\\\\\\\\\\\\\\\\\\'*10)
# confusion_matrix

from sklearn.metrics import confusion_matrix
confusion_matrix(y_test, yhat1)

# Build lightgbm model

In [ ]:
X_train_lgb = X_train.rename(columns = lambda x:re.sub('[^A-Za-z0-9_]+', '', x))
X_test_lgb = X_test.rename(columns = lambda x:re.sub('[^A-Za-z0-9_]+', '', x))

In [ ]:
# lgb_params

SEED = 42

lgb_params = {
    'bagging_freq': 7,
    'bagging_fraction': 0.9,
    'boost_from_average':'false',
    'boost': 'gbdt',
    'feature_fraction': 0.6,
    'learning_rate': 0.45,
    'max_depth': 13,
    'metric':'multi_logloss',
    #'min_data_in_leaf': 80,
    #'min_sum_hessian_in_leaf': 10.0,
    'num_leaves': 195,
    'num_threads': 25,
    'tree_learner': 'serial',
    'objective': 'multiclass', 
    'verbosity': 1
}

import lightgbm as lgb
clf2 = lgb.LGBMClassifier(**lgb_params)


# Fit model
clf2.fit(X_train_lgb, y_train)



# predict the results
yhat2 = clf2.predict(X_test_lgb)

# view accuracy
accuracy = accuracy_score(yhat2, y_test)
print('LightGBM Model accuracy score: {0:0.4f}'.format(accuracy_score(y_test, yhat2)))

print('//////////'*10)
print('\\\\\\\\\\\\\\\\\\\\'*10)

# confusion_matrix
confusion_matrix(y_test, yhat2)

# CatBoostClassifier

In [ ]:
from catboost import CatBoostClassifier


clf3 = CatBoostClassifier(iterations=100, learning_rate=0.07, l2_leaf_reg=3.5, 
                           depth=15, rsm=0.98, loss_function= 'MultiClass', eval_metric='Accuracy', 
                           metric_period=20, use_best_model=True,random_seed=42)


clf3.fit(X_train, y_train, eval_set=(X_test, y_test))


# predict the results
yhat3 = clf3.predict(X_test)


# view accuracy
accuracy = accuracy_score(yhat3, y_test)
print('CatBoostClassifier Model accuracy score: {0:0.4f}'.format(accuracy_score(y_test, yhat3)))

# Keras Neural Networks and Deep Learning

In [ ]:
train = pd.merge(df, titled_values, on='Inspection ID')
train = pd.merge(train, counts, on='Inspection ID')

In [ ]:
train = pd.concat( [train, pd.get_dummies(train['Results']),
                   pd.get_dummies(train['Risk'])] , axis = 1)
print("All Data Shape: ", train.shape)

In [ ]:
train.loc[:, train.columns.str.contains('Risk')].head()

In [ ]:
train = train.drop(['Inspection ID', 'DBA Name', 'AKA Name', 'Facility Type', 'City', 'Address', 
                    'State', 'Inspection Date', 'Inspection Type', 'Violations', 'Location', 
                    'Zip', 'Results', 'Risk', 'previous_results', 'datetime'], axis = 1)

In [ ]:
# create dataset
X = train.drop(['Risk 1 (High)', 'Risk 2 (Medium)', 'Risk 3 (Low)'], axis=1)
y = train[['Risk 1 (High)', 'Risk 2 (Medium)', 'Risk 3 (Low)']]
# split into train test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)
print(X_train.shape, X_test.shape, y_train.shape, y_test.shape)

In [ ]:
from tensorflow import keras
from tensorflow.keras import layers

model = keras.Sequential([
    layers.BatchNormalization(input_shape=[57]),
    layers.Dense(512, activation='relu'), 
    layers.BatchNormalization(),
    layers.Dropout(rate=0.5),
    layers.Dense(256, activation='relu'), 
    layers.BatchNormalization(),
    layers.Dropout(rate=0.5),
    layers.Dense(3, activation='sigmoid'),
])

In [ ]:
model.summary()

In [ ]:
model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy'],
)

In [ ]:
early_stopping = keras.callbacks.EarlyStopping(
    patience=10,
    min_delta=0.001,
    restore_best_weights=True,
)

history = model.fit(
    X_train, y_train,
    validation_data=(X_test, y_test),
    batch_size=256,
    epochs=100,
    callbacks=[early_stopping],
    verbose=1, # hide the output because we have so many epochs
)

In [ ]:
history_df = pd.DataFrame(history.history)
history_df.loc[:, ['loss', 'val_loss']].plot(title="Cross-entropy")
history_df.loc[:, ['accuracy', 'val_accuracy']].plot(title="Accuracy")

print(("Best Validation Loss: {:0.4f}" +\
      "\nBest Validation Accuracy: {:0.4f}")\
      .format(history_df['val_loss'].min(), 
              history_df['val_accuracy'].max()))

# XGBoost prform better than other models

In [ ]:
result['Prediction'] = yhat1

In [ ]:
result.head()

## We will select only predictions with Risk 1 (High) and Risk 2 (Medium) because they maximize revenue

In [ ]:
result = result[result['Prediction']!=2]

## Sort the results in descending  order based on the Prediction column

In [ ]:
result = result.sort_values('Prediction', ascending=False)

In [ ]:
result['Prediction'] = result['Prediction'].astype(str)
result['Name'] = result.apply(lambda row: row['AKA Name'] if not pd.isnull(row['AKA Name']) else row['DBA Name'], axis = 1)
risk_color_map = { "0": "rgb(255, 0, 0)", "1": "rgb(204, 204, 0)"}
result['Risk Color'] = result['Prediction'].map(risk_color_map)
#result.reset_index(inplace=True)
print("Total businesses: {}".format(result.shape[0]))

# Show suspected locations in the map with Risks High and Medium

In [ ]:
data = [
    go.Scattermapbox(
        lat = result['Latitude'],
        lon = result['Longitude'],
        text = result['Name'],
        hoverinfo = 'text',
        mode = 'markers',
        marker = go.scattermapbox.Marker(
            color = result['Risk Color'],
            opacity = 0.7,
            size = 4
        )
    )
]

layout = go.Layout(
    mapbox = dict(
        accesstoken = mapbox_access_token,
        zoom = 10,
        center = dict(
            lat = 41.8781,
            lon = -87.6298
        ),
    ),
    height = 800,
    width = 800,
    title = "Facilities in Chicago")

fig = go.Figure(data, layout)
iplot(fig, filename = 'facilities')


# Because we have only 12 inspectors, we can create a fuction that will show them next 12 inspectations with location and assign them tasks

In [ ]:
data = [
    go.Scattermapbox(
        lat = result['Latitude'],
        lon = result['Longitude'],
        text = result['Name'],
        hoverinfo = 'text',
        mode = 'markers',
        marker = go.scattermapbox.Marker(
            color = result['Risk Color'],
            opacity = 0.7,
            size = 4
        )
    )
]

layout = go.Layout(
    mapbox = dict(
        accesstoken = mapbox_access_token,
        zoom = 10,
        center = dict(
            lat = 41.8781,
            lon = -87.6298
        ),
    ),
    height = 800,
    width = 800,
    title = "Facilities in Chicago")

fig = go.Figure(data, layout)
iplot(fig, filename = 'facilities')


In [ ]:
def next_inpectations(result, n):
    for i in range(n):
        parts = result.iloc[i*12:12*(i+1),:]
        data = [
            go.Scattermapbox(
                lat = parts['Latitude'],
                lon = parts['Longitude'],
                text = parts['Name'],
                hoverinfo = 'text',
                mode = 'markers',
                marker = go.scattermapbox.Marker(
                    color = parts['Risk Color'],
                    opacity = 0.7,
                    size = 10
                )
            )
        ]

        layout = go.Layout(
            mapbox = dict(
                accesstoken = mapbox_access_token,
                zoom = 10,
                center = dict(
                    lat = 41.8781,
                    lon = -87.6298
                ),
            ),
            height = 800,
            width = 800,
            title = "Facilities in Chicago")

        fig = go.Figure(data, layout)
        iplot(fig, filename = 'facilities')

# The next 5 inspectations by the 12 inspectors (60 inspectations in totall).

In [ ]:
next_inpectations(result, 5)

## Finally, we can do some hyperparameters tuning,ensembling between models, add weather data, give inspectors the shortest path between facilities, so they can win time and money by visiting many facilities, and reducing gas consumption.